# Training ERM 

The experiment uses the same processed data and `MAX_LENGTH = 128` representation as the existing controlled ERM vs Group DRO comparison.


In [ ]:
#from google.colab import drive
#drive.mount("/content/drive")

from pathlib import Path
import os
#PROJECT_ROOT=Path("/content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments")
#if not PROJECT_ROOT.exists(): raise FileNotFoundError(PROJECT_ROOT)
#os.chdir(PROJECT_ROOT)
#print("Project root:",Path.cwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments


In [10]:
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_from_disk
from scipy.special import softmax
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

OUTPUT_DIR = Path("results/erm")
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


## Load and verify processed datasets


In [11]:
train_dataset = load_from_disk("data/processed/tokenized_train")
val_dataset = load_from_disk("data/processed/tokenized_validation")
test_dataset = load_from_disk("data/processed/tokenized_test")

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

IDENTITY_COLUMNS = [
    "male", "female", "LGBTQ", "christian",
    "muslim", "other_religions", "black", "white"
]

required = {"input_ids", "attention_mask", "label"} | set(IDENTITY_COLUMNS)
for name, dataset in [("Train",train_dataset),("Validation",val_dataset),("Test",test_dataset)]:
    missing = required - set(dataset.column_names)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")

print("Processed datasets verified.")


Train: 267516
Validation: 45047
Test: 132730
Processed datasets verified.


## Load tokenizer and fresh ERM model


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

print("Fresh ERM model loaded:", MODEL_NAME)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh ERM model loaded: distilbert-base-uncased


## Evaluation metrics


In [13]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
    }


##  Training section


In [15]:
# Calculate warmup steps manually to avoid the TrainingArguments argument issue
total_steps = (len(train_dataset) // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer configured with manual warmup_steps:", warmup_steps)

Trainer configured with manual warmup_steps: 5015


In [16]:
train_result = trainer.train()

print("ERM training complete.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation metric:", trainer.state.best_metric)

history_df = pd.DataFrame(trainer.state.log_history)
history_path = OUTPUT_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False)

print("Training history saved to:", history_path.resolve())
display(history_df.tail(20))


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.220939,0.190009,0.923591,0.684174,0.587406,0.632108
2,0.168773,0.192143,0.926987,0.736002,0.540524,0.623296


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.220939,0.190009,0.923591,0.684174,0.587406,0.632108
2,0.168773,0.192143,0.926987,0.736002,0.540524,0.623296
3,0.131209,0.261959,0.923103,0.675851,0.599325,0.635292


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ERM training complete.
Best checkpoint: results/erm/checkpoints/checkpoint-50160
Best validation metric: 0.6352916403453358
Training history saved to: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/erm/training_history.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_precision,eval_recall,eval_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,0.220939,0.825564,1.481493e-05,1.0,16720,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,1.0,16720,0.190009,0.923591,0.684174,0.587406,0.632108,151.9754,296.410,9.265,NaN,NaN,NaN,NaN,NaN
2,0.168773,0.604295,7.407686e-06,2.0,33440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,33440,0.192143,0.926987,0.736002,0.540524,0.623296,150.1763,299.961,9.376,NaN,NaN,NaN,NaN,NaN
4,0.131209,22.478214,4.430169e-10,3.0,50160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,3.0,50160,0.261959,0.923103,0.675851,0.599325,0.635292,150.6585,299.001,9.346,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,3.0,50160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8933.7297,89.833,5.615,2.651287e+16,0.17364


In [17]:
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

print("Best ERM model saved to:", BEST_MODEL_DIR.resolve())
print("Saved files:")
for path in sorted(BEST_MODEL_DIR.iterdir()):
    if path.is_file():
        print(" -", path.name)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best ERM model saved to: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/erm/best_model
Saved files:
 - config.json
 - model.safetensors
 - tokenizer (1).json
 - tokenizer.json
 - tokenizer_config (1).json
 - tokenizer_config.json
 - training_args.bin


## Test predictions


In [18]:
test_output = trainer.predict(test_dataset)

test_logits = test_output.predictions
test_labels = np.asarray(test_dataset["label"])
test_predictions = np.argmax(test_logits, axis=-1)
test_probabilities = softmax(test_logits, axis=1)

print("Test predictions generated:", len(test_predictions))


Test predictions generated: 132730


## Test metrics


In [19]:
test_metrics = {
    "accuracy": accuracy_score(test_labels, test_predictions),
    "precision": precision_score(test_labels, test_predictions, zero_division=0),
    "recall": recall_score(test_labels, test_predictions, zero_division=0),
    "f1": f1_score(test_labels, test_predictions, zero_division=0),
}

print("ERM test metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {value:.4f}")


ERM test metrics:
accuracy: 0.9227
precision: 0.6817
recall: 0.6014
f1: 0.6390


## Calculate all 16 identity-label groups and worst-group accuracy


In [20]:
rows=[]
for identity in IDENTITY_COLUMNS:
    identity_values = np.asarray(test_dataset[identity])
    for label_value in [0,1]:
        mask = (identity_values == 1) & (test_labels == label_value)
        n = int(mask.sum())
        accuracy = np.nan if n == 0 else accuracy_score(
            test_labels[mask], test_predictions[mask]
        )
        rows.append({
            "identity": identity,
            "label": label_value,
            "label_name": "non-toxic" if label_value == 0 else "toxic",
            "n": n,
            "accuracy": accuracy,
        })

group_results = pd.DataFrame(rows)
worst_group_accuracy = float(group_results["accuracy"].dropna().min())
test_metrics["worst_group_accuracy"] = worst_group_accuracy

print("Worst-group accuracy:", f"{worst_group_accuracy:.4f}")
print("Worst evaluated ERM group:")
display(group_results.sort_values("accuracy").head(1))


Worst-group accuracy: 0.4841
Worst evaluated ERM group:


,identity,label,label_name,n,accuracy
7,christian,1,toxic,1254,0.484051


In [21]:
prediction_df = test_dataset.to_pandas()
prediction_df["true_label"] = test_labels
prediction_df["prediction"] = test_predictions
prediction_df["prob_non_toxic"] = test_probabilities[:,0]
prediction_df["prob_toxic"] = test_probabilities[:,1]

prediction_path = OUTPUT_DIR / "test_predictions.csv"
prediction_df.to_csv(prediction_path, index=False)

print("ERM test predictions saved to:", prediction_path.resolve())
print("Shape:", prediction_df.shape)
print("Columns:", prediction_df.columns.tolist())


ERM test predictions saved to: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/erm/test_predictions.csv
Shape: (132730, 29)
Columns: ['uid', 'id', 'comment_text', 'toxicity', 'has_active_attrs', 'active_attrs', 'male', 'female', 'LGBTQ', 'christian', 'muslim', 'other_religions', 'black', 'white', 'identity_any', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'true_label', 'prediction', 'prob_non_toxic', 'prob_toxic']


In [22]:
metrics_path = OUTPUT_DIR / "test_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(
        {key: float(value) for key, value in test_metrics.items()},
        file,
        indent=4,
    )

group_path = OUTPUT_DIR / "test_group_metrics.csv"
group_results.to_csv(group_path, index=False)

print("Saved:", metrics_path.resolve())
print("Saved:", group_path.resolve())
print(json.dumps({key: float(value) for key,value in test_metrics.items()}, indent=2))


Saved: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/erm/test_metrics.json
Saved: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/erm/test_group_metrics.csv
{
  "accuracy": 0.9227378889474874,
  "precision": 0.6817362571342745,
  "recall": 0.6013911891354753,
  "f1": 0.6390482559571997,
  "worst_group_accuracy": 0.4840510366826156
}


In [23]:
experiment_config = {
    "model_name": MODEL_NAME,
    "algorithm": "ERM",
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "seed": SEED,
    "model_selection": "maximum validation F1",
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_f1": (
        float(trainer.state.best_metric)
        if trainer.state.best_metric is not None else None
    ),
    "evaluation_groups": "each positive identity crossed with toxicity label",
}

config_path = OUTPUT_DIR / "experiment_config.json"
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4)

print("Experiment configuration saved to:", config_path.resolve())


Experiment configuration saved to: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/erm/experiment_config.json


In [25]:
required_artifacts = [
    BEST_MODEL_DIR,
    OUTPUT_DIR / "test_predictions.csv",
    OUTPUT_DIR / "test_metrics.json",
    OUTPUT_DIR / "test_group_metrics.csv",
    OUTPUT_DIR / "training_history.csv",
    OUTPUT_DIR / "experiment_config.json",
]

artifact_check = pd.DataFrame([
    {"artifact": str(path), "exists": path.exists()}
    for path in required_artifacts
])
display(artifact_check)

missing = [str(path) for path in required_artifacts if not path.exists()]
if missing:
    raise FileNotFoundError("Missing ERM artifacts:\n" + "\n".join(missing))

print("All ERM artifacts saved successfully.")

,artifact,exists
0,results/erm/best_model,True
1,results/erm/test_predictions.csv,True
2,results/erm/test_metrics.json,True
3,results/erm/test_group_metrics.csv,True
4,results/erm/training_history.csv,True
5,results/erm/experiment_config.json,True


All ERM artifacts saved successfully.


In [26]:
saved = pd.read_csv(OUTPUT_DIR / "test_predictions.csv")

SELECTED_IDENTITY = "muslim"
SELECTED_LABEL = 1

subset = saved[
    (saved[SELECTED_IDENTITY] >= 0.5)
    & (saved["true_label"] == SELECTED_LABEL)
]

accuracy = (
    subset["prediction"] == subset["true_label"]
).mean()

print("Identity:", SELECTED_IDENTITY)
print("Label:", "toxic" if SELECTED_LABEL == 1 else "non-toxic")
print("Examples:", len(subset))
print("ERM accuracy:", accuracy)


Identity: muslim
Label: toxic
Examples: 1616
ERM accuracy: 0.5334158415841584


In [27]:
saved = pd.read_csv(OUTPUT_DIR / "test_predictions.csv")

SELECTED_IDENTITY = "christian"
SELECTED_LABEL = 1

subset = saved[
    (saved[SELECTED_IDENTITY] >= 0.5)
    & (saved["true_label"] == SELECTED_LABEL)
]

accuracy = (
    subset["prediction"] == subset["true_label"]
).mean()

print("Identity:", SELECTED_IDENTITY)
print("Label:", "toxic" if SELECTED_LABEL == 1 else "non-toxic")
print("Examples:", len(subset))
print("ERM accuracy:", accuracy)


Identity: christian
Label: toxic
Examples: 1254
ERM accuracy: 0.4840510366826156



- `best_model/`: reload the trained ERM model later without retraining.
- `test_predictions.csv`: perform identity-group, comparison, and error analyses without GPU/model loading.
- `test_metrics.json`: preserve final overall ERM metrics including worst-group accuracy.
- `test_group_metrics.csv`: preserve all 16 identity-label accuracies.
- `training_history.csv`: preserve training and validation logs.
- `experiment_config.json`: document exactly how the ERM experiment was produced.

